# ByteLLM — Comparaison contre Architecture Standard & Ablations (Google Colab)

Ce notebook compare scientifiquement à nombre de paramètres égal (~15M) :
1. **`vanilla`** : Transformeur d'octets standard (RoPE + SwiGLU pleine résolution, sans patching, sans bit-planes, sans convolutions locales, sans raisonnement récurrent)
2. **`bytellm` (complet)** : Architecture complète (`patch_size=4`, `reasoning_steps=3`, `bitplane=True`, `local_encoder=True`)
3. **Ablations composant par composant** : `reasoning_steps=1`, `patch_size=1`, `patch_size=8`, `--no-bitplane`, `--no-local-encoder`

In [ ]:
!rm -rf bytellm_repo && git clone https://github.com/cheikhoudieng/bytellm bytellm_repo
%cd bytellm_repo
!pip install -q -e .

## 1. Vérification rapide des variantes et de l'appariement des paramètres

In [ ]:
!python -c "from tests.test_bytellm import test_baseline_and_ablation_variants_are_causal_and_matched as t; t(); print('OK: toutes les variantes sont causales et appariees en parametres !')"

## 2. Benchmark : Transformeur Standard (`vanilla`) vs `ByteLLM` Complet + Ablations
Chaque run fait 3000 pas sur les mêmes 20 Mo d'enwik8 avec la même graine (`--seed 1`).

In [ ]:
# 1. Architecture Standard (Vanilla Byte Transformer apparié en paramètres)
!python -m bytellm.benchmark --arch vanilla --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --domains 0 --seed 1 --log-every 500 --out-dir bench_out/arch_cmp

# 2. ByteLLM Complet (patch=4, reasoning=3, bitplane, conv locale)
!python -m bytellm.benchmark --arch bytellm --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --reasoning-steps 3 --patch-size 4 --domains 0 --seed 1 --log-every 500 --out-dir bench_out/arch_cmp

# 3. Ablation : Sans Raisonnement Récurrent (reasoning_steps=1)
!python -m bytellm.benchmark --arch bytellm --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --reasoning-steps 1 --patch-size 4 --domains 0 --seed 1 --log-every 500 --out-dir bench_out/arch_cmp

# 4. Ablation : Sans Bit-Plane Embedding (--no-bitplane)
!python -m bytellm.benchmark --arch bytellm --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --reasoning-steps 3 --patch-size 4 --no-bitplane --domains 0 --seed 1 --log-every 500 --out-dir bench_out/arch_cmp

# 5. Ablation : Sans Encodeur Conv Local (--no-local-encoder)
!python -m bytellm.benchmark --arch bytellm --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --reasoning-steps 3 --patch-size 4 --no-local-encoder --domains 0 --seed 1 --log-every 500 --out-dir bench_out/arch_cmp

# 6. Ablation : Patch Size = 1 (pas de compression de patch) et Patch Size = 8
!python -m bytellm.benchmark --arch bytellm --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --reasoning-steps 3 --patch-size 1 --domains 0 --seed 1 --log-every 500 --out-dir bench_out/arch_cmp
!python -m bytellm.benchmark --arch bytellm --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --reasoning-steps 3 --patch-size 8 --domains 0 --seed 1 --log-every 500 --out-dir bench_out/arch_cmp

## 3. Tableau récapitulatif : Standard vs ByteLLM & Ablations

In [ ]:
import json, glob
files = sorted(glob.glob('bench_out/arch_cmp/results_*.json'))
rows = [json.load(open(f)) for f in files]
rows.sort(key=lambda r: r['test_bpb'])
print(f"{'Tag':38} | {'Params':>10} | {'Val BPB':>8} | {'Test BPB':>8} | {'Train(s)':>8} | {'Octets/s':>10}")
print('-'*96)
for r in rows:
    print(f"{r['tag']:38} | {r['parameters']:10,d} | {r['val_bpb']:8.4f} | {r['test_bpb']:8.4f} | {r['train_seconds']:8.1f} | {r['speed']['forward_bytes_per_sec']:10.0f}")
print('\nJSON brut a copier-coller :')
print(json.dumps([{k: r[k] for k in ('tag','parameters','val_bpb','test_bpb','train_seconds','speed')} for r in rows], indent=2))